In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By 
from selenium.webdriver.chrome.options import Options
import random
import time

# ===============================
# AYARLAR
# ===============================
BASLANGIC_SAYFASI = 7
BITIS_SAYFASI = 12

BASE_URL = "https://www.emlakjet.com/satilik-konut/izmir-bornova/?sayfa="
CIKTI_DOSYA = "bornova_7_12_sayfa_ilanlar.csv"


# ===============================
# SAYFA LİNKLERİNİ TOPLA
# ===============================
def ilan_linklerini_topla(driver, basla, bitir):
    linkler = set()

    for sayfa in range(basla, bitir + 1):
        url = BASE_URL + str(sayfa)
        print(f"\n➡ Sayfa {sayfa} çekiliyor: {url}")

        driver.get(url)
        time.sleep(random.uniform(4, 7))

        soup = BeautifulSoup(driver.page_source, "html.parser")
        cardlar = soup.find_all("a", href=True)

        sayfa_sayisi = 0
        for c in cardlar:
            if c["href"].startswith("/ilan/"):
                tam_link = "https://www.emlakjet.com" + c["href"]
                if tam_link not in linkler:
                    linkler.add(tam_link)
                    sayfa_sayisi += 1

        print(f"   -> Bu sayfada {sayfa_sayisi} yeni ilan bulundu | Toplam: {len(linkler)}")

    print(f"\n🎯 SONUÇ: {len(linkler)} adet benzersiz ilan linki toplandı.")
    return list(linkler)


# ===============================
# İLAN DETAY SAYFASINI ÇEK
# ===============================
def ilan_detaylarini_cek(driver, url):
    detaylar = {"URL": url}

    try:
        driver.get(url)
        time.sleep(random.uniform(4, 7))

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # FİYAT
        try:
            fiyat = soup.find("span", {"class": "styles_price__6wmxk"})
            detaylar["Fiyat"] = fiyat.text.strip() if fiyat else "N/A"
        except:
            detaylar["Fiyat"] = "N/A"

        # BAŞLIK
        try:
            baslik = soup.find("h1", class_="ej-title")
            detaylar["Baslik"] = baslik.text.strip() if baslik else "N/A"
        except:
            detaylar["Baslik"] = "N/A"

        # ADRES (İl - İlçe - Mahalle)
        try:
            adres = soup.find("span", class_="styles_location__HguIg")
            if adres:
                parcalar = adres.text.split("-")
                detaylar["İl"] = parcalar[0].strip() if len(parcalar) > 0 else "N/A"
                detaylar["İlçe"] = parcalar[1].strip() if len(parcalar) > 1 else "N/A"
                detaylar["Mahalle"] = parcalar[2].strip() if len(parcalar) > 2 else "N/A"
            else:
                detaylar["Mahalle"] = "N/A"
        except:
            detaylar["Mahalle"] = "N/A"

        # ÖZELLİKLER
        try:
            ozellik_div = soup.find("div", class_="styles_inner__qKPCB")
            if ozellik_div:
                for li in ozellik_div.find_all("li"):
                    spans = li.find_all("span")
                    if len(spans) >= 2:
                        key = spans[0].text.strip()
                        value = spans[-1].text.strip()
                        detaylar[key] = value
        except:
            pass

        # AÇIKLAMA
        try:
            aciklama_div = soup.find("div", class_="styles_wrapper__hycT_")
            detaylar["Aciklama"] = aciklama_div.text.strip().replace("\n", " ") if aciklama_div else "N/A"
        except:
            detaylar["Aciklama"] = "N/A"

    except Exception as e:
        print(f"   !!! HATA: {url} sayfası çekilemedi -> {e}")

    return detaylar


# ===============================
# ANA ÇALIŞMA BLOĞU
# ===============================
if __name__ == "__main__":
    print("🚀 Chrome başlatılıyor...")

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    # 1) LİNKLERİ TOPLA
    ilan_linkleri = ilan_linklerini_topla(driver, BASLANGIC_SAYFASI, BITIS_SAYFASI)

    # 2) DETAYLARI ÇEK
    tum_ilan_verileri = []
    print("\n🔍 İlan detayları çekiliyor...\n")

    for i, link in enumerate(ilan_linkleri):
        print(f"[{i+1}/{len(ilan_linkleri)}] {link}")
        detay = ilan_detaylarini_cek(driver, link)
        tum_ilan_verileri.append(detay)
        time.sleep(random.uniform(3, 6))

    # 3) CSV KAYDET
    if tum_ilan_verileri:
        df = pd.DataFrame(tum_ilan_verileri)
        df.to_csv(CIKTI_DOSYA, index=False, encoding="utf-8-sig")
        print(f"\n💾 VERİ KAYDEDİLDİ → {CIKTI_DOSYA}")
    else:
        print("❌ Veri yok, CSV oluşturulmadı.")

    driver.quit()
    print("🚪 Tarayıcı kapatıldı.")


🚀 Chrome başlatılıyor...

➡ Sayfa 7 çekiliyor: https://www.emlakjet.com/satilik-konut/izmir-bornova/?sayfa=7
   -> Bu sayfada 30 yeni ilan bulundu | Toplam: 30

➡ Sayfa 8 çekiliyor: https://www.emlakjet.com/satilik-konut/izmir-bornova/?sayfa=8
   -> Bu sayfada 30 yeni ilan bulundu | Toplam: 60

➡ Sayfa 9 çekiliyor: https://www.emlakjet.com/satilik-konut/izmir-bornova/?sayfa=9
   -> Bu sayfada 30 yeni ilan bulundu | Toplam: 90

➡ Sayfa 10 çekiliyor: https://www.emlakjet.com/satilik-konut/izmir-bornova/?sayfa=10
   -> Bu sayfada 30 yeni ilan bulundu | Toplam: 120

➡ Sayfa 11 çekiliyor: https://www.emlakjet.com/satilik-konut/izmir-bornova/?sayfa=11
   -> Bu sayfada 30 yeni ilan bulundu | Toplam: 150

➡ Sayfa 12 çekiliyor: https://www.emlakjet.com/satilik-konut/izmir-bornova/?sayfa=12
   -> Bu sayfada 30 yeni ilan bulundu | Toplam: 180

🎯 SONUÇ: 180 adet benzersiz ilan linki toplandı.

🔍 İlan detayları çekiliyor...

[1/180] https://www.emlakjet.com/ilan/bornova-buz-pistine-yuruyus-mesafesi

In [ ]:
df1=pd.read_csv("bornova_7_12_sayfa_ilanlar.csv")
df1.to_csv("C:\\Users\\ACER\\Desktop\\veriler3.csv", index=False, encoding="utf-8-sig")
pd.read_csv("C:\\Users\\ACER\\Desktop\\veriler3.csv").head()

,URL,Fiyat,Baslik,İl,İlçe,Mahalle,İlan Numarası,İlan Oluşturma Tarihi,İlan Güncelleme Tarihi,Türü,...,Banyo Metrekare,Balkon Metrekare,Salon Metrekare,WC Sayısı,WC Metrekare,Ada,Parsel,Görüntülü Gezilebilir mi?,Zemin Etüdü,Kattaki Daire Sayısı
0,https://www.emlakjet.com/ilan/bornova-buz-pist...,4.500.000 TL,NaN,İzmir,Bornova,Kızılay Mahallesi,18101492,21 Eylül 2025,27 Kasım 2025,Konut,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.emlakjet.com/ilan/yesilova-renova-...,6.800.000 TL,NaN,İzmir,Bornova,Yeşilova Mahallesi,18292202,NaN,24 Ekim 2025,Konut,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.emlakjet.com/ilan/bornova-merkeze-...,3.750.000 TL,NaN,İzmir,Bornova,Kızılay Mahallesi,18101512,21 Eylül 2025,27 Kasım 2025,Konut,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.emlakjet.com/ilan/krediye-uygun-21...,3.650.000 TL,NaN,İzmir,Bornova,Atatürk Mahallesi,18127682,NaN,30 Ekim 2025,Konut,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.emlakjet.com/ilan/turyap-tan-dogan...,5.750.000 TL,NaN,İzmir,Bornova,Doğanlar Mahallesi,18189978,NaN,09 Ekim 2025,Konut,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df1 = pd.read_csv("bornova_7_12_sayfa_ilanlar.csv")["İlan Numarası"]
df2 = pd.read_csv("C:\\Users\\ACER\\Desktop\\MlVERİLERİ\\veriler.csv")["İlan Numarası"]

df3 = pd.concat([df1, df2], ignore_index=True)

df3.nunique()

328